# Import thư viện

In [33]:
import pandas as pd

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.metrics import roc_auc_score

SEED = 42

In [34]:
train = pd.read_csv("./data/train.csv")
test = pd.read_csv("./data/test.csv")
TARGET = 'loan_paid_back'
print(train.shape)
print(test.shape)


(593994, 13)
(254569, 12)


In [35]:
X = train.drop(columns=['id', TARGET])
y = train[TARGET]

X_test = test.drop(columns=['id'])

num_cols = X.select_dtypes(exclude=['object', 'category']).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

print("Các cột dạng chữ:", cat_cols)
print("Các cột dạng số:", num_cols)

Các cột dạng chữ: ['gender', 'marital_status', 'education_level', 'employment_status', 'loan_purpose', 'grade_subgrade']
Các cột dạng số: ['annual_income', 'debt_to_income_ratio', 'credit_score', 'loan_amount', 'interest_rate']


In [36]:
X.isna().sum()

X[num_cols] = X[num_cols].fillna(X[num_cols].median())
X_test[num_cols] = X_test[num_cols].fillna(X_test[num_cols].median())

In [37]:
onehot = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', onehot, cat_cols)
    ],
    remainder='passthrough'
)
X_encode = preprocessor.fit_transform(X)
X_test_encode = preprocessor.transform(X_test)

X_train, X_val, y_train, y_val = train_test_split(X_encode, y, test_size=0.2, random_state=SEED)

print(f"Kích thước tập train: {len(X_train)}")
print(f"Kích thước tập val: {len(X_val)}")

Kích thước tập train: 475195
Kích thước tập val: 118799


In [38]:
logistic_regression = LogisticRegression(C=1.0, max_iter=1000, random_state=17)
decision_tree = DecisionTreeClassifier(max_depth=4, min_samples_split=2, min_samples_leaf=2, random_state=17)
random_forest = RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_split=2, min_samples_leaf=2, random_state=17)
xgboost = XGBClassifier(objective='binary:logistic', eval_metric='auc', n_estimator=5000, learning_rate=0.1, random_state=17)
lgbm = LGBMClassifier(objective='binary', metric='auc', n_estimators=5000, learning_rate=0.01, max_depth=4, random_state=17)

In [39]:
models = {
    'Logistic Regression': logistic_regression,
    'Decision Tree': decision_tree,
    'Random Forest': random_forest,
    'XGBoost': xgboost,
    'LightGBM': lgbm
}

for name, model in models.items():
    model.fit(X_train, y_train)
    val_probs = model.predict_proba(X_val)[:, 1]
    auc_score = roc_auc_score(y_val, val_probs)
    print(f"Model {name} đạt {auc_score}")

c:\Users\ADMIN\miniconda3\envs\dynamic\lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Model Logistic Regression đạt 0.8917977599073613
Model Decision Tree đạt 0.8895250639777931
Model Random Forest đạt 0.901048597835276


c:\Users\ADMIN\miniconda3\envs\dynamic\lib\site-packages\xgboost\core.py:158: UserWarning: [16:03:29] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "n_estimator" } are not used.

  warnings.warn(smsg, UserWarning)


Model XGBoost đạt 0.9185116726751452
[LightGBM] [Info] Number of positive: 379692, number of negative: 95503
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.009156 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1384
[LightGBM] [Info] Number of data points in the train set: 475195, number of used features: 60
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.799024 -> initscore=1.380203
[LightGBM] [Info] Start training from score 1.380203
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No furthe

c:\Users\ADMIN\miniconda3\envs\dynamic\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Model LightGBM đạt 0.9226459946593765


In [41]:
best_model = models["LightGBM"]
test_probs = best_model.predict_proba(X_test_encode)[:, 1]

submission = pd.read_csv("./data/sample_submission.csv")
submission[TARGET] = test_probs
submission.to_csv("./data/submission.csv", index=False)

c:\Users\ADMIN\miniconda3\envs\dynamic\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
